# Future Sales Prediction - Kaggle Competition

### Feature Engineering File

https://www.kaggle.com/competitions/competitive-data-science-predict-future-sales/

#### File descriptions
- sales_train.csv - the training set. Daily historical data from January 2013 to October 2015.

- test.csv - the test set. You need to forecast the sales for these shops and products for November 2015.

- sample_submission.csv - a sample submission file in the correct format.

- items.csv - supplemental information about the items/products.

- item_categories.csv  - supplemental information about the items categories.

- shops.csv- supplemental information about the shops.

#### Data fields

- ID - an Id that represents a (Shop, Item) tuple within the test set

- shop_id - unique identifier of a shop

- item_id - unique identifier of a product

- item_category_id - unique identifier of item category

- item_cnt_day - number of products sold. You are predicting a monthly amount of this measure

- item_price - current price of an item

- date - date in format dd/mm/yyyy

- date_block_num - a consecutive month number, used for convenience. January 2013 is 0, February 2013 is 1,..., October 2015 is 33

- item_name - name of item

- shop_name - name of shop

- item_category_name - name of item category

### I. Import the dataset and data analysis libraries

In [72]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [73]:
sales_train = pd.read_csv("data/sales_train.csv")
sales_train.head()

,date,date_block_num,shop_id,item_id,item_price,item_cnt_day
0,02.01.2013,0,59,22154,999.00,1.0
1,03.01.2013,0,25,2552,899.00,1.0
2,05.01.2013,0,25,2552,899.00,-1.0
3,06.01.2013,0,25,2554,1709.05,1.0
4,15.01.2013,0,25,2555,1099.00,1.0


In [74]:
test = pd.read_csv("data/test.csv")
test.head()

,ID,shop_id,item_id
0,0,5,5037
1,1,5,5320
2,2,5,5233
3,3,5,5232
4,4,5,5268


In [75]:
sample_submission = pd.read_csv("data/sample_submission.csv")
print(sample_submission.shape)
sample_submission.head()

(214200, 2)


,ID,item_cnt_month
0,0,0.5
1,1,0.5
2,2,0.5
3,3,0.5
4,4,0.5


In [76]:
items = pd.read_csv("data/items.csv")
items.head()

,item_name,item_id,item_category_id
0,! ВО ВЛАСТИ НАВАЖДЕНИЯ (ПЛАСТ.) D,0,40
1,!ABBYY FineReader 12 Professional Edition Full...,1,76
2,***В ЛУЧАХ СЛАВЫ (UNV) D,2,40
3,***ГОЛУБАЯ ВОЛНА (Univ) D,3,40
4,***КОРОБКА (СТЕКЛО) D,4,40


In [77]:
item_categories = pd.read_csv("data/item_categories.csv")
item_categories.head()

,item_category_name,item_category_id
0,PC - Гарнитуры/Наушники,0
1,Аксессуары - PS2,1
2,Аксессуары - PS3,2
3,Аксессуары - PS4,3
4,Аксессуары - PSP,4


In [78]:
shops = pd.read_csv("data/shops.csv")
shops.head()

,shop_name,shop_id
0,"!Якутск Орджоникидзе, 56 фран",0
1,"!Якутск ТЦ ""Центральный"" фран",1
2,"Адыгея ТЦ ""Мега""",2
3,"Балашиха ТРК ""Октябрь-Киномир""",3
4,"Волжский ТЦ ""Волга Молл""",4


### II. Initial data checks and manipulations

In [79]:
sales_train['date'] = pd.to_datetime(sales_train['date'], format = '%d.%m.%Y')
sales_train.head()

,date,date_block_num,shop_id,item_id,item_price,item_cnt_day
0,2013-01-02,0,59,22154,999.00,1.0
1,2013-01-03,0,25,2552,899.00,1.0
2,2013-01-05,0,25,2552,899.00,-1.0
3,2013-01-06,0,25,2554,1709.05,1.0
4,2013-01-15,0,25,2555,1099.00,1.0


In [80]:
# Split into month, day, and year
sales_train['year'] = sales_train['date'].dt.year
sales_train['month'] = sales_train['date'].dt.month
sales_train['day'] = sales_train['date'].dt.day
sales_train.head()

,date,date_block_num,shop_id,item_id,item_price,item_cnt_day,year,month,day
0,2013-01-02,0,59,22154,999.00,1.0,2013,1,2
1,2013-01-03,0,25,2552,899.00,1.0,2013,1,3
2,2013-01-05,0,25,2552,899.00,-1.0,2013,1,5
3,2013-01-06,0,25,2554,1709.05,1.0,2013,1,6
4,2013-01-15,0,25,2555,1099.00,1.0,2013,1,15


In [81]:
sales_train.isna().sum()

date              0
date_block_num    0
shop_id           0
item_id           0
item_price        0
item_cnt_day      0
year              0
month             0
day               0
dtype: int64

In [82]:
sales_train.nunique()

date               1034
date_block_num       34
shop_id              60
item_id           21807
item_price        19993
item_cnt_day        198
year                  3
month                12
day                  31
dtype: int64

In [83]:
# Describing datetime and quantitative variables
sales_train[['date', 'item_price', 'item_cnt_day']].describe()

,date,item_price,item_cnt_day
count,2935849,2.935849e+06,2.935849e+06
mean,2014-04-03 05:44:34.970681344,8.908532e+02,1.242641e+00
min,2013-01-01 00:00:00,-1.000000e+00,-2.200000e+01
25%,2013-08-01 00:00:00,2.490000e+02,1.000000e+00
50%,2014-03-04 00:00:00,3.990000e+02,1.000000e+00
75%,2014-12-05 00:00:00,9.990000e+02,1.000000e+00
max,2015-10-31 00:00:00,3.079800e+05,2.169000e+03
std,NaN,1.729800e+03,2.618834e+00


In [84]:
# Declaring categorical variables
sales_train['shop_id'] = pd.Categorical(sales_train['shop_id'])
sales_train['item_id'] = pd.Categorical(sales_train['item_id'])

In [85]:
# Convert negative to positive values in item_cnt_day
sales_train['item_cnt_day'] = sales_train['item_cnt_day'].abs()

In [86]:
# Create a "master table" for EDA in training data
sales_full = sales_train.merge(items, how = 'inner', on = 'item_id')\
                              .merge(item_categories, how = 'inner', on = 'item_category_id')\
                              .merge(shops, how = 'inner', on = 'shop_id')

sales_full.head()

,date,date_block_num,shop_id,item_id,item_price,item_cnt_day,year,month,day,item_name,item_category_id,item_category_name,shop_name
0,2013-01-02,0,59,22154,999.00,1.0,2013,1,2,ЯВЛЕНИЕ 2012 (BD),37,Кино - Blu-Ray,"Ярославль ТЦ ""Альтаир"""
1,2013-01-03,0,25,2552,899.00,1.0,2013,1,3,DEEP PURPLE The House Of Blue Light LP,58,Музыка - Винил,"Москва ТРК ""Атриум"""
2,2013-01-05,0,25,2552,899.00,1.0,2013,1,5,DEEP PURPLE The House Of Blue Light LP,58,Музыка - Винил,"Москва ТРК ""Атриум"""
3,2013-01-06,0,25,2554,1709.05,1.0,2013,1,6,DEEP PURPLE Who Do You Think We Are LP,58,Музыка - Винил,"Москва ТРК ""Атриум"""
4,2013-01-15,0,25,2555,1099.00,1.0,2013,1,15,DEEP PURPLE 30 Very Best Of 2CD (Фирм.),56,Музыка - CD фирменного производства,"Москва ТРК ""Атриум"""


#### Data checks:

1. Is there any item sold at more than one different price?

In [87]:
shops_and_items = sales_full[['shop_id', 'item_id', 'item_price']].copy()
shops_and_items.drop_duplicates(['shop_id', 'item_id'], inplace = True)

# Same shop, same item, different prices?
print(len(shops_and_items[shops_and_items.duplicated(subset = ['shop_id', 'item_id', 'item_price'])]))

0


2. Is there any group of items having different IDs but the same name? Do the same with shop name and item category name.

In [88]:
print(items.duplicated(['item_id']).sum()) # Number of duplicated item IDs
print(items.duplicated(['item_name']).sum()) # Number of duplicated item names

0
0


In [89]:
# With item categories
print(item_categories.duplicated(['item_category_id']).sum())
print(item_categories.duplicated(['item_category_name']).sum())

0
0


In [90]:
# With shops
print(shops.duplicated(['shop_id']).sum())
print(shops.duplicated(['shop_name']).sum())

0
0


3. Is there any day when the shop can't sell any item?

In [91]:
# Generate list of datetime indices - all days of sales in the training data
datetime_indices = sales_full[['date', 'date_block_num']].copy()
datetime_indices.drop_duplicates(inplace = True)
datetime_indices.sort_values('date', inplace = True)
datetime_indices.set_index('date', inplace = True)
print(datetime_indices.index)

DatetimeIndex(['2013-01-01', '2013-01-02', '2013-01-03', '2013-01-04',
               '2013-01-05', '2013-01-06', '2013-01-07', '2013-01-08',
               '2013-01-09', '2013-01-10',
               ...
               '2015-10-22', '2015-10-23', '2015-10-24', '2015-10-25',
               '2015-10-26', '2015-10-27', '2015-10-28', '2015-10-29',
               '2015-10-30', '2015-10-31'],
              dtype='datetime64[ns]', name='date', length=1034, freq=None)


In [92]:
# Generate a list of dates from the first and last day of sales in the training data
date_range = pd.date_range(start = sales_full['date'].min(), end = sales_full['date'].max())
print(date_range)

DatetimeIndex(['2013-01-01', '2013-01-02', '2013-01-03', '2013-01-04',
               '2013-01-05', '2013-01-06', '2013-01-07', '2013-01-08',
               '2013-01-09', '2013-01-10',
               ...
               '2015-10-22', '2015-10-23', '2015-10-24', '2015-10-25',
               '2015-10-26', '2015-10-27', '2015-10-28', '2015-10-29',
               '2015-10-30', '2015-10-31'],
              dtype='datetime64[ns]', length=1034, freq='D')


#### Fuzzy string matching - a part of data check:

Are there any pair of item names, shop names, or item category names close to each other? If this is possible, it might be the case of spelling or processing errors, or two categories which can be grouped to one. (Kaggle community said probably)

In [93]:
# https://typesense.org/learn/fuzzy-string-matching-python/
# https://pypi.org/project/fuzzywuzzy/
# https://towardsdatascience.com/a-fuzzy-string-matching-story-314bbecaa098/

from itertools import combinations, chain
from fuzzywuzzy import fuzz
from fuzzywuzzy import process

In [94]:
def fuzzy_string_matching_metrics(df, name_column):
    '''df: the DataFrame in question
    name_column: the column name that you want to check for fuzzy string matching (must be a string)'''

    name_list = df[name_column].tolist() # extract name pairs and convert them to a Python list
    pairwise_combinations = list(combinations(name_list, 2)) # generate all pairwise combinations of shop names to calculate similarity

    # https://stackoverflow.com/questions/7422453/python-change-type-of-whole-list
    # Change each element of the pairwise combinations into a list, then create a DataFrame for easier sorting
    pairwise_combinations = list(map(list, pairwise_combinations))
    pairwise_df = pd.DataFrame(pairwise_combinations, columns = ['first_string', 'second_string'])

    # Compute various similarity metrics
    pairwise_df['fuzz_ratio'] = pairwise_df.apply(lambda row: fuzz.ratio(row['first_string'], row['second_string']), axis = 1)
    pairwise_df['partial_ratio'] = pairwise_df.apply(lambda row: fuzz.partial_ratio(row['first_string'], row['second_string']), axis = 1)
    pairwise_df['token_sort_ratio'] = pairwise_df.apply(lambda row: fuzz.token_sort_ratio(row['first_string'], row['second_string']), axis = 1)
    pairwise_df['token_set_ratio'] = pairwise_df.apply(lambda row: fuzz.token_set_ratio(row['first_string'], row['second_string']), axis = 1)

    return pairwise_df

In [95]:
def sort_desc_with_threshold(df, sort_column, threshold):
    # Keep the necessary columns
    df_filtered = df[['first_string', 'second_string', sort_column]]

    # Filter for values greater or equal to a threshold 
    df_filtered = df_filtered[df_filtered[sort_column] >= threshold]

    # Sort values in descending order
    df_filtered.sort_values(sort_column, ascending = False, inplace = True)
    return df_filtered

In [96]:
def get_unique_values(*args):
    '''Pass individual list of lists as inputs. Returns a list of lists.'''

    # Combine the list of lists together and flatten it to be a 2D list
    whole_list = list(chain(args))
    flattened_list = [item for sublist in whole_list for item in sublist]

    # Convert each element of individual list to tuple format to later use the "set" data type (keeping unique values only)
    list_of_tuples = [tuple(x) for x in flattened_list]
    unique_val_set = set(list_of_tuples) # return a set object

    # Convert back into list of tuples and then list of lists
    unique_val_list = list(unique_val_set)
    unique_val_list = [list(x) for x in unique_val_list]

    return unique_val_list

In [97]:
def flag_similar_pairs(original_df, name_column, fuzz_ratio_threshold = 90, partial_ratio_threshold = 90,
                       token_sort_ratio_threshold = 90, token_set_ratio_threshold = 90):
    '''You can change the similarity thresholds by modifying the default arguments in this function.'''
    
    # Generate a DataFrame with fuzzy string matching metrics of all pairs of names in a particular DataFrame column
    df = fuzzy_string_matching_metrics(original_df, name_column)

    # Find similar pairs with high fuzz ratio, partial ratio, token sort ratio and token set ratio
    top_fuzz_ratios = sort_desc_with_threshold(df, 'fuzz_ratio', fuzz_ratio_threshold)[['first_string', 'second_string']].values.tolist()
    top_partial_ratios = sort_desc_with_threshold(df, 'partial_ratio', partial_ratio_threshold)[['first_string', 'second_string']].values.tolist()
    top_token_sort_ratios = sort_desc_with_threshold(df, 'token_sort_ratio', token_sort_ratio_threshold)[['first_string', 'second_string']].values.tolist()
    top_token_set_ratios = sort_desc_with_threshold(df, 'token_set_ratio', token_set_ratio_threshold)[['first_string', 'second_string']].values.tolist()

    # Get unique values that are high in at least one metric...
    unique_pairs_list = get_unique_values(top_fuzz_ratios, top_partial_ratios, top_token_sort_ratios, top_token_set_ratios)
    # ...then output a DataFrame for better readability
    result_df = df[df[['first_string', 'second_string']].apply(list, axis = 1).isin(unique_pairs_list)]
    return result_df

**For shop names:**

Because I have written so many functions, I decided to show the intermediate steps to arrive at finding similar pairs of names for this particular example.

In [98]:
shops.head()

,shop_name,shop_id
0,"!Якутск Орджоникидзе, 56 фран",0
1,"!Якутск ТЦ ""Центральный"" фран",1
2,"Адыгея ТЦ ""Мега""",2
3,"Балашиха ТРК ""Октябрь-Киномир""",3
4,"Волжский ТЦ ""Волга Молл""",4


In [99]:
# Example of a table of fuzzy string matching metrics
fuzzy_shop_names = fuzzy_string_matching_metrics(shops, 'shop_name')
fuzzy_shop_names.head()

,first_string,second_string,fuzz_ratio,partial_ratio,token_sort_ratio,token_set_ratio
0,"!Якутск Орджоникидзе, 56 фран","!Якутск ТЦ ""Центральный"" фран",52,52,53,59
1,"!Якутск Орджоникидзе, 56 фран","Адыгея ТЦ ""Мега""",22,33,29,29
2,"!Якутск Орджоникидзе, 56 фран","Балашиха ТРК ""Октябрь-Киномир""",20,21,33,33
3,"!Якутск Орджоникидзе, 56 фран","Волжский ТЦ ""Волга Молл""",23,25,33,33
4,"!Якутск Орджоникидзе, 56 фран","Вологда ТРЦ ""Мармелад""",24,28,34,34


In [100]:
# Example of keeping pairs with the highest fuzz ratio and sorting them in descending order
top_fuzz_ratio = sort_desc_with_threshold(fuzzy_shop_names, 'fuzz_ratio', 80)
top_fuzz_ratio

,first_string,second_string,fuzz_ratio
545,Жуковский ул. Чкалова 39м?,Жуковский ул. Чкалова 39м²,96
1104,"Москва ТК ""Буденовский"" (пав.А2)","Москва ТК ""Буденовский"" (пав.К7)",94
56,"!Якутск Орджоникидзе, 56 фран","Якутск Орджоникидзе, 56",88
115,"!Якутск ТЦ ""Центральный"" фран","Якутск ТЦ ""Центральный""",88
1560,"РостовНаДону ТРК ""Мегацентр Горизонт""","РостовНаДону ТРК ""Мегацентр Горизонт"" Островной",88
1335,"Москва ТЦ ""Перловский""","Москва ТЦ ""Семеновский""",84
1554,"Омск ТЦ ""Мега""","Химки ТЦ ""Мега""",83


In [101]:
# When the whole function is completed...
flag_similar_pairs(shops, 'shop_name')

,first_string,second_string,fuzz_ratio,partial_ratio,token_sort_ratio,token_set_ratio
56,"!Якутск Орджоникидзе, 56 фран","Якутск Орджоникидзе, 56",88,100,90,100
115,"!Якутск ТЦ ""Центральный"" фран","Якутск ТЦ ""Центральный""",88,100,89,100
545,Жуковский ул. Чкалова 39м?,Жуковский ул. Чкалова 39м²,96,96,100,100
1104,"Москва ТК ""Буденовский"" (пав.А2)","Москва ТК ""Буденовский"" (пав.К7)",94,94,89,94
1560,"РостовНаДону ТРК ""Мегацентр Горизонт""","РостовНаДону ТРК ""Мегацентр Горизонт"" Островной",88,100,88,100
1561,"РостовНаДону ТРК ""Мегацентр Горизонт""","РостовНаДону ТЦ ""Мега""",71,91,69,75
1580,"РостовНаДону ТРК ""Мегацентр Горизонт"" Островной","РостовНаДону ТЦ ""Мега""",61,91,58,75


There are some particularly interesting similar name combinations that have a high similarity (at least a similarity metric >= 90/100). I will use Google Translate to try to understand them.

- Index 56: ```!Якутск Орджоникидзе, 56 фран``` (!Yakutsk Ordzhonikidze, 56 fran) and ```Якутск Орджоникидзе, 56``` (Yakutsk Ordzhonikidze).

- Index 115: ```!Якутск ТЦ "Центральный" фран``` (!Yakutsk shopping center "Central" franc) and ```Якутск ТЦ "Центральный"``` (Yakutsk shopping center "Central").

- Index 545: ```Жуковский ул. Чкалова 39м?``` (Zhukovsky st. Chkalova 39m?) and ```Жуковский ул. Чкалова 39м²``` (Zhukovsky st. Chkalova 39m²).

- Index 1104: ```Москва ТК "Буденовский" (пав.А2)``` (Moscow TC "Budenovsky" (pavilion A2)) and ```Москва ТК "Буденовский" (пав.К7)``` (Moscow TC "Budenovsky" (pavement K7)).

- Index 1560, 1561 and 1580: ```РостовНаДону ТРК "Мегацентр Горизонт"``` (Rostov-on-Don TRC "Megacenter Gorizont"), ```РостовНаДону ТРК "Мегацентр Горизонт" Островной``` (Rostov-on-Don TRC "Megacenter Gorizont" Island) and ```РостовНаДону ТЦ "Мега"``` (Rostov-on-Don TC "Mega").

**Fuzzy string matching results:**

- ```фран``` (56, 115) does not have a clear meaning in Russian, so I think that this is a result of inaccurate processing or data entry.

- There are some unwanted special characters, such as ```!``` and ```?``` (56, 115, 545), that made spotting duplicated shop names impossible without fuzzy string matching.

- I have asked ChatGPT to save my time browsing through Russian sources while I don't understand anything about it, and it said that while ```РостовНаДону ТРК "Мегацентр Горизонт"``` and ```РостовНаДону ТРК "Мегацентр Горизонт" Островной``` (1560) refer to the same shop, ```РостовНаДону ТЦ "Мега"``` is another shopping center separated from these two names.

**For item category names:**

In [102]:
item_categories.head()

,item_category_name,item_category_id
0,PC - Гарнитуры/Наушники,0
1,Аксессуары - PS2,1
2,Аксессуары - PS3,2
3,Аксессуары - PS4,3
4,Аксессуары - PSP,4


In [103]:
flag_similar_pairs(item_categories, 'item_category_name')

,first_string,second_string,fuzz_ratio,partial_ratio,token_sort_ratio,token_set_ratio
83,Аксессуары - PS2,Аксессуары - PS3,94,94,93,93
84,Аксессуары - PS2,Аксессуары - PS4,94,94,93,93
85,Аксессуары - PS2,Аксессуары - PSP,94,94,93,93
86,Аксессуары - PS2,Аксессуары - PSVita,86,94,84,84
165,Аксессуары - PS3,Аксессуары - PS4,94,94,93,93
166,Аксессуары - PS3,Аксессуары - PSP,94,94,93,93
167,Аксессуары - PS3,Аксессуары - PSVita,86,94,84,84
246,Аксессуары - PS4,Аксессуары - PSP,94,94,93,93
247,Аксессуары - PS4,Аксессуары - PSVita,86,94,84,84
326,Аксессуары - PSP,Аксессуары - PSVita,86,94,84,84


**Decoding similar category names with Google Translate:**

| Russian      | English       | Russian   | English | Russian         | English       |
|--------------|---------------|-----------|---------|-----------------|---------------|
| Аксессуары   | Accessories   | Книги     | Books   | Игровые консоли | Game consoles |
| Подарки      | Gifts         | Игры      | Games   | Программы       | Programs      |
| Карты оплаты | Payment cards | Служебные | Utility | Кино            | Movies        |

**Special word combinations in Russian:**

- ```Книги - Аудиокниги (Цифра)```: Books - Audiobooks (Digital)

- ```Подарки - Настольные игры (компактные)```: Gifts - Board games (compact)

- ```Подарки - Сувениры (в навеску)```: Gifts - Souvenirs (as a gift)

- ```Программы - Для дома и офиса (Цифра)```: Programs - For home and office (Digital)

- ```Программы - Обучающие (Цифра)```: Programs - Educational (Digital)

- ```Служебные - Билеты```: Service - Tickets

**String similarity matching result:**

- Although being different types, ```Аксессуары``` (accessories), ```Игровые консоли``` (game consoles), and ```Игры``` (games) still have categories with the same names such as PS2, PS3, or PS4. For feature engineering, I can regroup each PS product into an item category.

- For the annotations (in the **Special word combinations** part):

    + ```Цифра``` (digital), ```компактные``` (compact), and ```Билеты``` (tickets) are additional information to distinguish between different item categories. I have decided to keep them as a separate category.

    + Because ```Подарки``` is already "gift" in Russian, the additional annotation ```в навеску``` (as a gift) is only extra information. I will merge ```Подарки - Сувениры``` and ```Подарки - Сувениры (в навеску)``` into a single category.

Python outputs ```MemoryError``` when I was working with item names (>20,000 different names), but I think that cleaning the shop names and item category names is enough.

#### Applying fuzzy string matching results to the sales dataset:

In [104]:
# Stripping sales_full of item, shop, and item category names
sales_full.drop(['item_name', 'item_category_name', 'shop_name'], axis = 1, inplace = True)
sales_full.head()

,date,date_block_num,shop_id,item_id,item_price,item_cnt_day,year,month,day,item_category_id
0,2013-01-02,0,59,22154,999.00,1.0,2013,1,2,37
1,2013-01-03,0,25,2552,899.00,1.0,2013,1,3,58
2,2013-01-05,0,25,2552,899.00,1.0,2013,1,5,58
3,2013-01-06,0,25,2554,1709.05,1.0,2013,1,6,58
4,2013-01-15,0,25,2555,1099.00,1.0,2013,1,15,56


**Cleaning shop data:**

In [105]:
shops[shops['shop_name'].isin(['!Якутск Орджоникидзе, 56 фран', 'Якутск Орджоникидзе, 56'])]

,shop_name,shop_id
0,"!Якутск Орджоникидзе, 56 фран",0
57,"Якутск Орджоникидзе, 56",57


In [106]:
shops[shops['shop_name'].isin(['!Якутск ТЦ "Центральный" фран', 'Якутск ТЦ "Центральный"'])]

,shop_name,shop_id
1,"!Якутск ТЦ ""Центральный"" фран",1
58,"Якутск ТЦ ""Центральный""",58


In [107]:
shops[shops['shop_name'].isin(['Жуковский ул. Чкалова 39м?', 'Жуковский ул. Чкалова 39м²'])]

,shop_name,shop_id
10,Жуковский ул. Чкалова 39м?,10
11,Жуковский ул. Чкалова 39м²,11


In [108]:
shops[shops['shop_name'].isin(['РостовНаДону ТРК "Мегацентр Горизонт"', 'РостовНаДону ТРК "Мегацентр Горизонт" Островной'])]

,shop_name,shop_id
39,"РостовНаДону ТРК ""Мегацентр Горизонт""",39
40,"РостовНаДону ТРК ""Мегацентр Горизонт"" Островной",40


In [109]:
# Change the necessary shop IDs
sales_full['shop_id'] = sales_full['shop_id'].copy().replace(0, 57)
sales_full['shop_id'] = sales_full['shop_id'].copy().replace(1, 58)
sales_full['shop_id'] = sales_full['shop_id'].copy().replace(10, 11)
sales_full['shop_id'] = sales_full['shop_id'].copy().replace(40, 39)

sales_full.head()

,date,date_block_num,shop_id,item_id,item_price,item_cnt_day,year,month,day,item_category_id
0,2013-01-02,0,59,22154,999.00,1.0,2013,1,2,37
1,2013-01-03,0,25,2552,899.00,1.0,2013,1,3,58
2,2013-01-05,0,25,2552,899.00,1.0,2013,1,5,58
3,2013-01-06,0,25,2554,1709.05,1.0,2013,1,6,58
4,2013-01-15,0,25,2555,1099.00,1.0,2013,1,15,56


**Cleaning item category data:**

In [110]:
item_categories.head()

,item_category_name,item_category_id
0,PC - Гарнитуры/Наушники,0
1,Аксессуары - PS2,1
2,Аксессуары - PS3,2
3,Аксессуары - PS4,3
4,Аксессуары - PSP,4


In [111]:
item_categories[item_categories['item_category_name'].str.contains('PS')]

,item_category_name,item_category_id
1,Аксессуары - PS2,1
2,Аксессуары - PS3,2
3,Аксессуары - PS4,3
4,Аксессуары - PSP,4
5,Аксессуары - PSVita,5
10,Игровые консоли - PS2,10
11,Игровые консоли - PS3,11
12,Игровые консоли - PS4,12
13,Игровые консоли - PSP,13
14,Игровые консоли - PSVita,14


In [112]:
item_categories[item_categories['item_category_name'].str.contains('Подарки - Сувениры')]

,item_category_name,item_category_id
69,Подарки - Сувениры,69
70,Подарки - Сувениры (в навеску),70


In [113]:
# Change the necessary item category IDs
sales_full['item_category_id'] = sales_full['item_category_id'].copy().replace(10, 1) # for PS2
sales_full['item_category_id'] = sales_full['item_category_id'].copy().replace(18, 1)

sales_full['item_category_id'] = sales_full['item_category_id'].copy().replace(11, 2) # for PS3
sales_full['item_category_id'] = sales_full['item_category_id'].copy().replace(19, 2)

sales_full['item_category_id'] = sales_full['item_category_id'].copy().replace(12, 3) # for PS4
sales_full['item_category_id'] = sales_full['item_category_id'].copy().replace(20, 3)

sales_full['item_category_id'] = sales_full['item_category_id'].copy().replace(13, 4) # for PSP
sales_full['item_category_id'] = sales_full['item_category_id'].copy().replace(21, 4)

sales_full['item_category_id'] = sales_full['item_category_id'].copy().replace(14, 5) # for PSVita
sales_full['item_category_id'] = sales_full['item_category_id'].copy().replace(22, 5)

sales_full['item_category_id'] = sales_full['item_category_id'].copy().replace(70, 69) # for "Подарки - Сувениры"

sales_full.head()

,date,date_block_num,shop_id,item_id,item_price,item_cnt_day,year,month,day,item_category_id
0,2013-01-02,0,59,22154,999.00,1.0,2013,1,2,37
1,2013-01-03,0,25,2552,899.00,1.0,2013,1,3,58
2,2013-01-05,0,25,2552,899.00,1.0,2013,1,5,58
3,2013-01-06,0,25,2554,1709.05,1.0,2013,1,6,58
4,2013-01-15,0,25,2555,1099.00,1.0,2013,1,15,56


### III. Feature engineering

Here, we consider ```date_block_num``` variable as number of months elapsed after January 2013, and this column can also be useful for splitting into different sets.

Aggregate existing features:

In [114]:
# Keep the necessary columns only - we are predicting by month
sales_features = sales_full[['year', 'month', 'date_block_num', 'shop_id', 'item_category_id', 'item_id', 'item_price', 'item_cnt_day']].copy()

# Because no item has more than 1 price, we can group item price by the median
sales_features_price = sales_features[['year', 'month', 'date_block_num', 'shop_id', 'item_category_id', 'item_id', 'item_price']]\
    .groupby(['year', 'month', 'date_block_num', 'shop_id', 'item_category_id', 'item_id'], as_index = False).median()

# Group item count by month by the sum
sales_features_itemcnt = sales_features[['year', 'month', 'date_block_num', 'shop_id', 'item_category_id', 'item_id', 'item_cnt_day']]\
    .groupby(['year', 'month', 'date_block_num', 'shop_id', 'item_category_id', 'item_id'], as_index = False).sum()

sales_features_itemcnt.rename(columns = {'item_cnt_day': 'item_cnt_month'}, inplace = True)

sales_features = sales_features_price.merge(sales_features_itemcnt, how = 'inner', on = ['year', 'month', 'date_block_num', 'shop_id', 'item_category_id', 'item_id'])
sales_features.sort_values(['year', 'month', 'date_block_num', 'shop_id', 'item_category_id', 'item_id', 'item_price', 'item_cnt_month'], inplace = True)
sales_features.head()

,year,month,date_block_num,shop_id,item_category_id,item_id,item_price,item_cnt_month
0,2013,1,0,2,2,27,2499.0,1.0
1,2013,1,0,2,2,1409,1398.5,1.0
2,2013,1,0,2,2,1467,899.0,1.0
3,2013,1,0,2,2,1471,2599.0,2.0
4,2013,1,0,2,2,1832,1999.0,1.0


Create a separate table to match ```date_block_num``` with months and years:

In [115]:
# Guide for date block number
date_block = sales_features[['year', 'month', 'date_block_num']].copy().drop_duplicates(ignore_index = True)
date_block.head()

,year,month,date_block_num
0,2013,1,0
1,2013,2,1
2,2013,3,2
3,2013,4,3
4,2013,5,4


Number of unique items in each shop/category for each month.

Median price of each shop/category/item.

In [116]:
# Number of unique items and median price in each shop

shops_and_items = sales_features[['year', 'month', 'shop_id', 'item_price', 'item_id']].copy()

shops_and_items = shops_and_items.groupby(['year', 'month', 'shop_id'], as_index = False)\
    .agg({'item_price': 'median', 'item_id': 'nunique'})
shops_and_items.rename(columns = {'item_price': 'med_price_shop', 'item_id': 'unique_items_in_shop'}, inplace = True)

shops_and_items.head()

,year,month,shop_id,med_price_shop,unique_items_in_shop
0,2013,1,2,399.0,728
1,2013,1,3,399.0,544
2,2013,1,4,299.0,1062
3,2013,1,6,349.0,1865
4,2013,1,7,349.0,1271


In [117]:
# Number of unique items and median price in each item category

cat_and_items = sales_features[['year', 'month', 'item_category_id', 'item_price', 'item_id']].copy()

cat_and_items = cat_and_items.groupby(['year', 'month', 'item_category_id'], as_index = False)\
    .agg({'item_price': 'median', 'item_id': 'nunique'})
cat_and_items.rename(columns = {'item_price': 'med_price_cat', 'item_id': 'unique_items_in_cat'}, inplace = True)

cat_and_items.head()

,year,month,item_category_id,med_price_cat,unique_items_in_cat
0,2013,1,0,148.0,1
1,2013,1,1,398.0,2
2,2013,1,2,1499.0,289
3,2013,1,3,499.0,2
4,2013,1,4,599.0,96


In [118]:
# Monthly median price of all items sold in the month in general

med_price_items = sales_features[['year', 'month', 'item_price']].copy()

med_price_items = med_price_items.groupby(['year', 'month'], as_index = False).median()
med_price_items.rename(columns = {'item_price': 'med_price_all_items'}, inplace = True)

med_price_items.head()

,year,month,med_price_all_items
0,2013,1,299.0
1,2013,2,299.0
2,2013,3,299.0
3,2013,4,299.0
4,2013,5,299.0


Create weekend and holiday features:

In [119]:
# Separate the sales days together
sales_days = sales_full[['date']].copy()
sales_days.drop_duplicates(inplace = True, ignore_index = True)
sales_days.sort_values('date', inplace = True, ignore_index = True)

# Create month, year, and weekend indicator variables
sales_days['year'] = sales_days['date'].dt.year
sales_days['month'] = sales_days['date'].dt.month
sales_days['weekend_indicator'] = sales_days['date'].case_when(
    [(sales_days['date'].dt.day_of_week.isin([5, 6]), 1),
     (~sales_days['date'].dt.day_of_week.isin([5, 6]), 0)]
)

sales_days.head(7)

,date,year,month,weekend_indicator
0,2013-01-01,2013,1,0
1,2013-01-02,2013,1,0
2,2013-01-03,2013,1,0
3,2013-01-04,2013,1,0
4,2013-01-05,2013,1,1
5,2013-01-06,2013,1,1
6,2013-01-07,2013,1,0


In [120]:
# See Analysis File for more details
# List of holidays in Russia: https://www.timeanddate.com/holidays/russia/2013 (same as for 2014 and 2015)
holiday_list_2013 = ['2013-01-01', '2013-01-02', '2013-01-03', '2013-01-04', '2013-01-05', '2013-01-06', '2013-01-07', '2013-01-08',
                     '2013-02-23', '2013-03-08', '2013-05-01', '2013-05-02', '2013-05-03', '2013-05-09', '2013-05-10',
                     '2013-06-12', '2013-11-04']

holiday_list_2014 = ['2014-01-01', '2014-01-02', '2014-01-03', '2014-01-06', '2014-01-07', '2014-01-08',
                     '2014-02-22', '2014-02-23', '2014-03-08', '2014-03-09', '2014-03-10', '2014-05-01', '2014-05-02',
                     '2014-05-03', '2014-05-09', '2014-05-10', '2014-05-11', '2014-06-12', '2014-06-13', '2014-06-14', '2014-06-15',
                     '2014-11-01', '2014-11-02', '2014-11-03', '2014-11-04']

holiday_list_2015 = ['2015-01-01', '2015-01-02', '2015-01-03', '2015-01-04', '2015-01-05', '2015-01-06', '2015-01-07',
                     '2015-01-08', '2015-01-09', '2015-02-23', '2015-03-08', '2015-03-09', '2015-05-01', '2015-05-04', '2015-05-09',
                     '2015-05-11', '2015-06-12'] # only count until October 2015

# Combine to have a list of holidays in training data
holidays = holiday_list_2013 + holiday_list_2014 + holiday_list_2015
holidays = [pd.Timestamp(x) for x in holidays] # Convert the whole list to timestamp format

# Create holiday indicator variable
sales_days['holiday_indicator'] = sales_days['date'].case_when(
    [(sales_days['date'].isin(holidays), 1),
     (~sales_days['date'].isin(holidays), 0)]
)

sales_days.head()

,date,year,month,weekend_indicator,holiday_indicator
0,2013-01-01,2013,1,0,1
1,2013-01-02,2013,1,0,1
2,2013-01-03,2013,1,0,1
3,2013-01-04,2013,1,0,1
4,2013-01-05,2013,1,1,1


In [121]:
# Count the number of days as weekends and holidays in a (month, year) pair
sales_days.drop('date', axis = 1, inplace = True)
sales_month = sales_days.groupby(['year', 'month'], as_index = False).sum()
sales_month.rename(columns = {'weekend_indicator': 'weekends_in_month', 'holiday_indicator': 'holidays_in_month'}, inplace = True)
sales_month.head(7)

,year,month,weekends_in_month,holidays_in_month
0,2013,1,8,8
1,2013,2,8,1
2,2013,3,10,1
3,2013,4,8,0
4,2013,5,8,5
5,2013,6,10,1
6,2013,7,8,0


Create seasonal features:

In [122]:
sales_month['season'] = sales_month['month'].case_when(
    [(sales_month['month'].isin([3, 4, 5]), 'spring'),
     (sales_month['month'].isin([6, 7, 8]), 'summer'),
     (sales_month['month'].isin([9, 10, 11]), 'fall'),
     (sales_month['month'].isin([12, 1, 2]), 'winter')]
)

# Because winter is very cold in Russia, I will encode so that winter goes first
sales_month['season'] = pd.Categorical(values = sales_month['season'], ordered = True,
                                       categories = ['winter', 'spring', 'summer', 'fall'])

# Use one-hot encoding to turn the seasons into features
# Drop the first column (winter) to avoid multicollinearity
onehot_season = pd.get_dummies(sales_month['season'], drop_first = True)

# Convert data type of all the one-hot encoded values: True/False to 0/1
onehot_season = onehot_season.map(lambda x: int(x))

# Merge the one-hot encoding into the sales_month feature DataFrame
sales_month = pd.concat([sales_month, onehot_season], axis = 1)

# Drop the categorical 'season' column now that we have 3 one-hot encoded season columns
sales_month.drop('season', axis = 1, inplace = True)

sales_month.head(7)

,year,month,weekends_in_month,holidays_in_month,spring,summer,fall
0,2013,1,8,8,0,0,0
1,2013,2,8,1,0,0,0
2,2013,3,10,1,1,0,0
3,2013,4,8,0,1,0,0
4,2013,5,8,5,1,0,0
5,2013,6,10,1,0,1,0
6,2013,7,8,0,0,1,0


Possible feature: number of months in which the item was sold to date

Create lag features:

Here, we consider ```date_block_num``` variable as number of months elapsed after January 2013, and this column can also be useful for splitting into different sets.

In [123]:
# Create revenue features
# Revenue = Price * Item count
sales_features['revenue'] = sales_features['item_price'] * sales_features['item_cnt_month']
sales_features.head()

,year,month,date_block_num,shop_id,item_category_id,item_id,item_price,item_cnt_month,revenue
0,2013,1,0,2,2,27,2499.0,1.0,2499.0
1,2013,1,0,2,2,1409,1398.5,1.0,1398.5
2,2013,1,0,2,2,1467,899.0,1.0,899.0
3,2013,1,0,2,2,1471,2599.0,2.0,5198.0
4,2013,1,0,2,2,1832,1999.0,1.0,1999.0


In [124]:
# Thanks to ChatGPT, I now get to know a resampling method with respect to multiple indices

def resampling_by_date_block(df, grouped_col_name):
    '''The DataFrame should have 3 columns named: date_block_num, a grouped column (item_id/ item_category_id/ shop_id),
    and a value column of your choice (such as revenue) that is always the rightmost column.'''

    # Create a list of all possible (id, date_block_num) pairs (from_product here refers to Cartesian product between 2 sets)
    # Including months that the item/category/shop couldn't sell any items
    full_index = pd.MultiIndex.from_product([df[grouped_col_name].unique(), np.arange(34)], names = [grouped_col_name, 'date_block_num'])

    # Resampling the (id, date_block_num) pairs
    # A bit of a roundabout when I have to convert them to index and then back to normal columns
    df = df.set_index([grouped_col_name, 'date_block_num'], drop = True)
    df = df.reindex(full_index).reset_index()

    # For the pairs not included in the original dataset, I impute with 0 (as in the case of revenue)
    df = df.fillna(0)

    return df

In [125]:
# Create lag features for revenue across each item
# https://tichmangono.github.io/tutorials/2018/05/04/time-series-data-munging-lagging-variables-across-multiple-groups

def lagged_in_group(df, grouped_col_name, periods = 1, new_col_name = 'lagged'):
    '''The DataFrame used as input should have the same expected input format as the resampling_by_date_block function.
    Outputs a new DataFrame.
    "periods" denote the number of lagged periods (default = 1).'''

    # Shift by item_id across date_block_num
    df_lagged = df.set_index(['date_block_num', grouped_col_name])
    df_lagged = df_lagged.unstack().shift(periods = periods)

    # Restore the unstacked indices to its original order
    df_lagged = df_lagged.stack(future_stack = True)
    df_lagged = df_lagged.reset_index()

    # Additional formatting: filling NaN values with 0 and change column names
    df_lagged = df_lagged.fillna(0)
    df_lagged = df_lagged.rename(columns = {df_lagged.columns[2]: new_col_name})

    return df_lagged

In [126]:
# Example to check: item 22167 in the "tail" of the revenue dataset
sales_features[np.logical_and(sales_features['item_id'] == 22167, sales_features['date_block_num'] == 33)]

,year,month,date_block_num,shop_id,item_category_id,item_id,item_price,item_cnt_month,revenue
1579239,2015,10,33,6,49,22167,299.0,1.0,299.0
1580851,2015,10,33,11,49,22167,155.0,1.0,155.0
1581345,2015,10,33,12,49,22167,299.0,7.0,2093.0
1583848,2015,10,33,18,49,22167,299.0,1.0,299.0
1585875,2015,10,33,21,49,22167,299.0,1.0,299.0
1586684,2015,10,33,22,49,22167,299.0,14.0,4186.0
1588723,2015,10,33,25,49,22167,299.0,3.0,897.0
1590240,2015,10,33,26,49,22167,299.0,1.0,299.0
1591307,2015,10,33,28,49,22167,299.0,1.0,299.0
1595682,2015,10,33,37,49,22167,299.0,1.0,299.0


Monthly item revenue:

In [127]:
# Aggregate monthly item revenue
sales_rev = sales_features[['date_block_num', 'item_id', 'revenue']]
sales_rev = sales_rev.groupby(['date_block_num', 'item_id'], as_index = False).sum()
sales_rev.sort_values(['item_id', 'date_block_num'], inplace = True)
sales_rev.rename(columns = {'revenue': 'monthly_item_revenue'}, inplace = True)

sales_rev.head()

,date_block_num,item_id,monthly_item_revenue
153402,20,0,58.0
120288,15,1,8980.0
140428,18,1,4490.0
147039,19,1,4490.0
153403,20,1,4490.0


In [128]:
# Example to check: item 22167 in the "tail" of the revenue dataset
sales_rev[sales_rev['item_id'] == 22167].tail()

,date_block_num,item_id,monthly_item_revenue
212982,29,22167,9723.0
218305,30,22167,10166.0
223413,31,22167,8671.0
228498,32,22167,6279.0
233911,33,22167,10919.0


In [129]:
sales_rev_resampled = resampling_by_date_block(sales_rev, 'item_id')
sales_rev_resampled.head()

,item_id,date_block_num,monthly_item_revenue
0,0,0,0.0
1,0,1,0.0
2,0,2,0.0
3,0,3,0.0
4,0,4,0.0


In [130]:
# 1-month lag of total item revenue
sales_rev_lagged_1m = lagged_in_group(sales_rev_resampled, 'item_id', periods = 1, new_col_name = 'item_revenue_lagged_1m')
sales_rev_lagged_1m.tail()

,date_block_num,item_id,item_revenue_lagged_1m
741433,33,22165,0.0
741434,33,22166,750.0
741435,33,22167,6279.0
741436,33,22168,0.0
741437,33,22169,0.0


In [131]:
# 3-month lag of total item revenue
sales_rev_lagged_3m = lagged_in_group(sales_rev_resampled, 'item_id', periods = 3, new_col_name = 'item_revenue_lagged_3m')
sales_rev_lagged_3m.tail()

,date_block_num,item_id,item_revenue_lagged_3m
741433,33,22165,0.0
741434,33,22166,1200.0
741435,33,22167,10166.0
741436,33,22168,0.0
741437,33,22169,0.0


Merge the lagged features into the monthly item revenue DataFrame:

In [132]:
sales_rev_final = sales_rev_resampled.merge(sales_rev_lagged_1m, on = ['item_id', 'date_block_num'], how = 'left')\
    .merge(sales_rev_lagged_3m, on = ['item_id', 'date_block_num'], how = 'left')

sales_rev_final[sales_rev_final['item_id'] == 22167].tail()

,item_id,date_block_num,monthly_item_revenue,item_revenue_lagged_1m,item_revenue_lagged_3m
741365,22167,29,9723.0,9867.0,11960.0
741366,22167,30,10166.0,9723.0,11362.0
741367,22167,31,8671.0,10166.0,9867.0
741368,22167,32,6279.0,8671.0,9723.0
741369,22167,33,10919.0,6279.0,10166.0


Using the same method, create lagged revenue features by shops and item categories:

In [133]:
# Aggregate monthly shop revenue
shops_rev = sales_features[['date_block_num', 'shop_id', 'revenue']]
shops_rev = shops_rev.groupby(['date_block_num', 'shop_id'], as_index = False).sum()
shops_rev.sort_values(['shop_id', 'date_block_num'], inplace = True)
shops_rev.rename(columns = {'revenue': 'monthly_shop_revenue'}, inplace = True)

shops_rev.head()

,date_block_num,shop_id,monthly_shop_revenue
0,0,2,1.097687e+06
45,1,2,5.255847e+05
91,2,2,7.348829e+05
137,3,2,6.167043e+05
183,4,2,5.428603e+05


In [134]:
# Resampling and adding lag features for shop revenue
shops_rev_resampled = resampling_by_date_block(shops_rev, 'shop_id')

shops_rev_lagged_1m = lagged_in_group(shops_rev_resampled, 'shop_id', periods = 1, new_col_name = 'shop_revenue_lagged_1m')
shops_rev_lagged_3m = lagged_in_group(shops_rev_resampled, 'shop_id', periods = 3, new_col_name = 'shop_revenue_lagged_3m')


shops_rev_final = shops_rev_resampled.merge(shops_rev_lagged_1m, on = ['shop_id', 'date_block_num'], how = 'left')\
    .merge(shops_rev_lagged_3m, on = ['shop_id', 'date_block_num'], how = 'left')

shops_rev_final.tail()

,shop_id,date_block_num,monthly_shop_revenue,shop_revenue_lagged_1m,shop_revenue_lagged_3m
1899,59,29,8.609416e+05,1.022668e+06,1.026213e+06
1900,59,30,8.725792e+05,8.609416e+05,1.049967e+06
1901,59,31,9.870302e+05,8.725792e+05,1.022668e+06
1902,59,32,1.110135e+06,9.870302e+05,8.609416e+05
1903,59,33,9.725480e+05,1.110135e+06,8.725792e+05


In [135]:
# Aggregate monthly category revenue
category_rev = sales_features[['date_block_num', 'item_category_id', 'revenue']]
category_rev = category_rev.groupby(['date_block_num', 'item_category_id'], as_index = False).sum()
category_rev.sort_values(['item_category_id', 'date_block_num'], inplace = True)
category_rev.rename(columns = {'revenue': 'monthly_cat_revenue'}, inplace = True)

category_rev.tail()

,date_block_num,item_category_id,monthly_cat_revenue
1586,29,83,33962.000
1641,30,83,38846.380
1698,31,83,37971.735
1754,32,83,38079.000
1810,33,83,45537.440


In [136]:
# Resampling and adding lag features for category revenue
category_rev_resampled = resampling_by_date_block(category_rev, 'item_category_id')

category_rev_lagged_1m = lagged_in_group(category_rev_resampled, 'item_category_id', periods = 1, new_col_name = 'cat_revenue_lagged_1m')
category_rev_lagged_3m = lagged_in_group(category_rev_resampled, 'item_category_id', periods = 3, new_col_name = 'cat_revenue_lagged_3m')

category_rev_final = category_rev_resampled.merge(category_rev_lagged_1m, on = ['item_category_id', 'date_block_num'], how = 'left')\
    .merge(category_rev_lagged_3m, on = ['item_category_id', 'date_block_num'], how = 'left')

category_rev_final.tail()

,item_category_id,date_block_num,monthly_cat_revenue,cat_revenue_lagged_1m,cat_revenue_lagged_3m
2477,83,29,33962.000,35493.000,69463.00
2478,83,30,38846.380,33962.000,36774.00
2479,83,31,37971.735,38846.380,35493.00
2480,83,32,38079.000,37971.735,33962.00
2481,83,33,45537.440,38079.000,38846.38


Now combine the features together:

In [164]:
# Add the initial features
monthly_features = shops_and_items\
    .merge(cat_and_items, how = 'inner', on = ['year', 'month'])\
    .merge(med_price_items, how = 'inner', on = ['year', 'month'])\
    .merge(sales_month, how = 'inner', on = ['year', 'month'])\

monthly_features.head(7)

,year,month,shop_id,med_price_shop,unique_items_in_shop,item_category_id,med_price_cat,unique_items_in_cat,med_price_all_items,weekends_in_month,holidays_in_month,spring,summer,fall
0,2013,1,2,399.0,728,0,148.00,1,299.0,8,8,0,0,0
1,2013,1,2,399.0,728,1,398.00,2,299.0,8,8,0,0,0
2,2013,1,2,399.0,728,2,1499.00,289,299.0,8,8,0,0,0
3,2013,1,2,399.0,728,3,499.00,2,299.0,8,8,0,0,0
4,2013,1,2,399.0,728,4,599.00,96,299.0,8,8,0,0,0
5,2013,1,2,399.0,728,5,1699.00,74,299.0,8,8,0,0,0
6,2013,1,2,399.0,728,6,1493.75,19,299.0,8,8,0,0,0


In [165]:
# Add the date_block_num column
date_block_features = monthly_features.merge(date_block, how = 'left', on = ['year', 'month'])
date_block_features.head(7)

,year,month,shop_id,med_price_shop,unique_items_in_shop,item_category_id,med_price_cat,unique_items_in_cat,med_price_all_items,weekends_in_month,holidays_in_month,spring,summer,fall,date_block_num
0,2013,1,2,399.0,728,0,148.00,1,299.0,8,8,0,0,0,0
1,2013,1,2,399.0,728,1,398.00,2,299.0,8,8,0,0,0,0
2,2013,1,2,399.0,728,2,1499.00,289,299.0,8,8,0,0,0,0
3,2013,1,2,399.0,728,3,499.00,2,299.0,8,8,0,0,0,0
4,2013,1,2,399.0,728,4,599.00,96,299.0,8,8,0,0,0,0
5,2013,1,2,399.0,728,5,1699.00,74,299.0,8,8,0,0,0,0
6,2013,1,2,399.0,728,6,1493.75,19,299.0,8,8,0,0,0,0


In [166]:
# Add item_id, item_price, item_cnt_month, etc. data

more_item_features = sales_features.merge(date_block_features, how = 'left', on = ['year', 'month', 'date_block_num', 'shop_id', 'item_category_id'])
more_item_features.tail(7)

,year,month,date_block_num,shop_id,item_category_id,item_id,item_price,item_cnt_month,revenue,med_price_shop,unique_items_in_shop,med_price_cat,unique_items_in_cat,med_price_all_items,weekends_in_month,holidays_in_month,spring,summer,fall
1608265,2015,10,33,59,75,4178,1590.0,3.0,4770.0,449.0,500,1590.0,46,499.0,9,0,0,0,1
1608266,2015,10,33,59,75,4181,1290.0,7.0,9030.0,449.0,500,1590.0,46,499.0,9,0,0,0,1
1608267,2015,10,33,59,75,5383,4390.0,1.0,4390.0,449.0,500,1590.0,46,499.0,9,0,0,0,1
1608268,2015,10,33,59,79,17717,500.0,13.0,6500.0,449.0,500,999.0,1,499.0,9,0,0,0,1
1608269,2015,10,33,59,83,22087,119.0,6.0,714.0,449.0,500,119.0,4,499.0,9,0,0,0,1
1608270,2015,10,33,59,83,22088,119.0,2.0,238.0,449.0,500,119.0,4,499.0,9,0,0,0,1
1608271,2015,10,33,59,83,22091,179.0,1.0,179.0,449.0,500,119.0,4,499.0,9,0,0,0,1


In [167]:
# Add the lag features
final_features = more_item_features\
    .merge(sales_rev_final, how = 'left', on = ['date_block_num', 'item_id'])\
    .merge(shops_rev_final, how = 'left', on = ['date_block_num', 'shop_id'])\
    .merge(category_rev_final, how = 'left', on = ['date_block_num', 'item_category_id'])

# Display all columns in the DataFrame, now that we have many features
pd.set_option('display.max_columns', None)
final_features.tail(7)

,year,month,date_block_num,shop_id,item_category_id,item_id,item_price,item_cnt_month,revenue,med_price_shop,unique_items_in_shop,med_price_cat,unique_items_in_cat,med_price_all_items,weekends_in_month,holidays_in_month,spring,summer,fall,monthly_item_revenue,item_revenue_lagged_1m,item_revenue_lagged_3m,monthly_shop_revenue,shop_revenue_lagged_1m,shop_revenue_lagged_3m,monthly_cat_revenue,cat_revenue_lagged_1m,cat_revenue_lagged_3m
1608265,2015,10,33,59,75,4178,1590.0,3.0,4770.0,449.0,500,1590.0,46,499.0,9,0,0,0,1,175486.000,180243.280000,162653.28,972548.0,1.110135e+06,872579.25,1632346.740,1.329378e+06,1176059.53
1608266,2015,10,33,59,75,4181,1290.0,7.0,9030.0,449.0,500,1590.0,46,499.0,9,0,0,0,1,286608.000,235972.000000,214405.40,972548.0,1.110135e+06,872579.25,1632346.740,1.329378e+06,1176059.53
1608267,2015,10,33,59,75,5383,4390.0,1.0,4390.0,449.0,500,1590.0,46,499.0,9,0,0,0,1,74830.000,10780.000000,0.00,972548.0,1.110135e+06,872579.25,1632346.740,1.329378e+06,1176059.53
1608268,2015,10,33,59,79,17717,500.0,13.0,6500.0,449.0,500,999.0,1,499.0,9,0,0,0,1,533156.625,505494.166667,445328.40,972548.0,1.110135e+06,872579.25,533156.625,5.054942e+05,445328.40
1608269,2015,10,33,59,83,22087,119.0,6.0,714.0,449.0,500,119.0,4,499.0,9,0,0,0,1,10353.000,7243.000000,10849.38,972548.0,1.110135e+06,872579.25,45537.440,3.807900e+04,38846.38
1608270,2015,10,33,59,83,22088,119.0,2.0,238.0,449.0,500,119.0,4,499.0,9,0,0,0,1,22015.000,15299.000000,16683.00,972548.0,1.110135e+06,872579.25,45537.440,3.807900e+04,38846.38
1608271,2015,10,33,59,83,22091,179.0,1.0,179.0,449.0,500,119.0,4,499.0,9,0,0,0,1,5228.000,7661.000000,5299.00,972548.0,1.110135e+06,872579.25,45537.440,3.807900e+04,38846.38


Some last preprocessing:

In [168]:
# Encode month using trigonometric encoding for cyclical features
final_features['month_sin'] = np.sin(final_features['month'] / 12 * 2 * np.pi)
final_features['month_cos'] = np.cos(final_features['month'] / 12 * 2 * np.pi)

# Drop the unnecessary variables
# I can't just reproduce the current revenue variables in the test set because we are predicting sales for the current month
final_features.drop(['month', 'revenue', 'monthly_item_revenue', 'monthly_shop_revenue', 'monthly_cat_revenue'], axis = 1, inplace = True)

final_features.head(7)

,year,date_block_num,shop_id,item_category_id,item_id,item_price,item_cnt_month,med_price_shop,unique_items_in_shop,med_price_cat,unique_items_in_cat,med_price_all_items,weekends_in_month,holidays_in_month,spring,summer,fall,item_revenue_lagged_1m,item_revenue_lagged_3m,shop_revenue_lagged_1m,shop_revenue_lagged_3m,cat_revenue_lagged_1m,cat_revenue_lagged_3m,month_sin,month_cos
0,2013,0,2,2,27,2499.0,1.0,399.0,728,1499.0,289,299.0,8,8,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.866025
1,2013,0,2,2,1409,1398.5,1.0,399.0,728,1499.0,289,299.0,8,8,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.866025
2,2013,0,2,2,1467,899.0,1.0,399.0,728,1499.0,289,299.0,8,8,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.866025
3,2013,0,2,2,1471,2599.0,2.0,399.0,728,1499.0,289,299.0,8,8,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.866025
4,2013,0,2,2,1832,1999.0,1.0,399.0,728,1499.0,289,299.0,8,8,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.866025
5,2013,0,2,2,2281,499.5,2.0,399.0,728,1499.0,289,299.0,8,8,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.866025
6,2013,0,2,2,2307,2799.0,2.0,399.0,728,1499.0,289,299.0,8,8,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.866025


In [169]:
final_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1608272 entries, 0 to 1608271
Data columns (total 25 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   year                    1608272 non-null  int32  
 1   date_block_num          1608272 non-null  int64  
 2   shop_id                 1608272 non-null  int64  
 3   item_category_id        1608272 non-null  int64  
 4   item_id                 1608272 non-null  int64  
 5   item_price              1608272 non-null  float64
 6   item_cnt_month          1608272 non-null  float64
 7   med_price_shop          1608272 non-null  float64
 8   unique_items_in_shop    1608272 non-null  int64  
 9   med_price_cat           1608272 non-null  float64
 10  unique_items_in_cat     1608272 non-null  int64  
 11  med_price_all_items     1608272 non-null  float64
 12  weekends_in_month       1608272 non-null  object 
 13  holidays_in_month       1608272 non-null  object 
 14  sp

In [170]:
previous_usage = final_features.memory_usage(deep = True).sum()
print(previous_usage)

405284676


In [171]:
final_features.describe().loc[['min', 'max']]

,year,date_block_num,shop_id,item_category_id,item_id,item_price,item_cnt_month,med_price_shop,unique_items_in_shop,med_price_cat,unique_items_in_cat,med_price_all_items,spring,summer,fall,item_revenue_lagged_1m,item_revenue_lagged_3m,shop_revenue_lagged_1m,shop_revenue_lagged_3m,cat_revenue_lagged_1m,cat_revenue_lagged_3m,month_sin,month_cos
min,2013.0,0.0,2.0,0.0,0.0,0.09,1.0,77.0,1.0,0.5,1.0,299.0,0.0,0.0,0.0,0.000000e+00,0.000000e+00,0.00,0.00,0.000000e+00,0.000000e+00,-1.0,-1.0
max,2015.0,33.0,59.0,83.0,22169.0,307980.00,2253.0,7990.0,3939.0,28490.0,2540.0,499.0,1.0,1.0,1.0,4.563052e+07,4.563052e+07,15639178.92,15639178.92,6.879167e+07,6.879167e+07,1.0,1.0


In [183]:
# Optimize data types
final_features['year'] = final_features['year'].astype('int16')
final_features['date_block_num'] = final_features['date_block_num'].astype('uint8')
final_features['shop_id'] = final_features['shop_id'].astype('uint8')
final_features['item_category_id'] = final_features['item_category_id'].astype('uint8')
final_features['item_id'] = final_features['item_id'].astype('uint16')
final_features['item_price'] = final_features['item_price'].astype('float32')
final_features['item_cnt_month'] = final_features['item_cnt_month'].astype('float16')
final_features['med_price_shop'] = final_features['med_price_shop'].astype('float16')
final_features['unique_items_in_shop'] = final_features['unique_items_in_shop'].astype('uint16')
final_features['med_price_cat'] = final_features['med_price_shop'].astype('float16')
final_features['unique_items_in_cat'] = final_features['unique_items_in_shop'].astype('uint16')
final_features['med_price_all_items'] = final_features['med_price_all_items'].astype('float16')
final_features['spring'] = final_features['spring'].astype('uint8')
final_features['summer'] = final_features['summer'].astype('uint8')
final_features['fall'] = final_features['fall'].astype('uint8')
final_features['month_sin'] = final_features['month_sin'].astype('float16')
final_features['month_cos'] = final_features['month_cos'].astype('float16')

final_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1608272 entries, 0 to 1608271
Data columns (total 25 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   year                    1608272 non-null  int16  
 1   date_block_num          1608272 non-null  uint8  
 2   shop_id                 1608272 non-null  uint8  
 3   item_category_id        1608272 non-null  uint8  
 4   item_id                 1608272 non-null  uint16 
 5   item_price              1608272 non-null  float32
 6   item_cnt_month          1608272 non-null  float16
 7   med_price_shop          1608272 non-null  float16
 8   unique_items_in_shop    1608272 non-null  uint16 
 9   med_price_cat           1608272 non-null  float16
 10  unique_items_in_cat     1608272 non-null  uint16 
 11  med_price_all_items     1608272 non-null  float16
 12  weekends_in_month       1608272 non-null  object 
 13  holidays_in_month       1608272 non-null  object 
 14  sp

In [184]:
current_usage = final_features.memory_usage(deep = True).sum()
print(f'Current memory usage of final_features DataFrame: {current_usage}')
print(f'Compression: approx. {round((current_usage / previous_usage * 100), 2)}%')

Current memory usage of final_features DataFrame: 241240932
Compression: approx. 59.52%


#### Explanation of new feature names (work later):

- ```med_price_shop```, ```unique_items_in_shop```: Median price of an item and number of unique items sold in a particular shop in the month

- ```med_price_cat```, ```unique_items_in_cat```: Median price of an item and number of unique items belonging to a particular category sold in the month

- ```med_price_all_items```: Median price of all items sold in the month

- ```weekends_in_month```: Number of days of weekend (Saturday or Sunday) in the month

- ```holidays_in_month```: Number of days of public holiday in the month

- ```spring```, ```summer```, ```fall```: Dummy variables denoting the season that an item was sold. If ```spring = summer = fall = 0```, then this item was sold in winter

- ```month_sin```, ```month_cos```: Month in a year, encoded using trigonometric encoding to reflect its cyclical nature throughout a year.

In [185]:
# Export the final training data with features for later use in the 3rd file
final_features.to_csv('final_features.csv', index = False)

### Useful resources and references:

- https://www.kaggle.com/code/thnhnguyntrngtt/predict-future-sales-0-86-solution (partly written in Vietnamese - LGBM with early stopping)

- https://www.kaggle.com/code/cngnguynt04/lightgbm-0-87442 (contains many useful tips to optimize performance)